In [1]:
import dendropy
import treeswift as ts
import msprime
import numpy as np
import collections
import sys
import tskit
import demes

# import daiquiri
# daiquiri.setup(level="DEBUG")

In [2]:
tns = dendropy.TaxonNamespace()
t = dendropy.Tree.get(
    path="../main_neoaves-num_generations.tre",
    schema="newick",
    preserve_underscores=True,
    taxon_namespace=tns,
)

In [3]:
samples = []
height = t.max_distance_from_root()
name_to_pop = {}
for ix, node in enumerate(t.leaf_nodes()):
    # st = height - node.distance_from_root()
    # node.edge_length += st
    st = 0
    samples.append(msprime.SampleSet(1, ploidy=1, time=st, population=node.taxon.label))
    name_to_pop[ix] = node.taxon.label

In [4]:
initial_size = {}
with open("../population-sizes.tsv", "r") as f:
    initial_size = dict(map(lambda x: x.strip().split("\t"), f.readlines()[1:]))
initial_size = {k: float(v) / 2 for k, v in initial_size.items()}

In [5]:
tree = t.as_string(schema="newick", suppress_rooting=True, unquoted_underscores=True)
demography = msprime.Demography.from_species_tree(tree, initial_size)
demography.sort_events()

In [6]:
demes_path = "../anomaly_cases/demography_admixture-050_N349_050_N299_to_Strigiformes.yaml"
graph = demes.load(demes_path)
demography = msprime.Demography.from_demes(graph)
demography.sort_events()

In [7]:
SL = 10000
R = 5e-9

In [49]:
events = [e for e in demography.events]
time_cencus = []
migrated_pairs = []
for e in events:
    edict = e.asdict()
    if isinstance(edict.get("ancestral", ""), list):
        print(edict)
        time_cencus.append((edict["time"]))
        # demography.add_census(edict["time"] + 1e-7)
        # demography.add_census(edict["time"] - 1e-7)
        for pa in e.ancestral:
            migrated_pairs.append((pa, edict["derived"]))
demography.sort_events()

{'time': 7728512.195121953, 'derived': 'Strigiformes', 'ancestral': ['N349', 'N299'], 'proportions': [0.5, 0.5]}


In [9]:
migrated_pairs

[('N349', 'Strigiformes'), ('N299', 'Strigiformes')]

In [10]:
time_cencus

[7728512.195121953]

In [12]:
tmain = ts.read_tree_newick("../main_neoaves-num_generations.tre")
label_to_node = tmain.label_to_node(selection="all")
nd_interest_set = set()
for a, d in migrated_pairs:
    nd_interest_set.add(label_to_node[a])
    nd_interest_set.add(label_to_node[d])
parent_interest_set = set()
for nd in nd_interest_set:
    nd_p = nd
    while nd_p is not None:
        parent_interest_set.add(nd_p.get_label())
        nd_p = nd_p.get_parent()

In [108]:
b_interest = "N299"

In [109]:
tmain = ts.read_tree_newick("../main_neoaves-num_generations.tre")
label_to_node = tmain.label_to_node(selection="all")
nd_interest = label_to_node[b_interest]
population_interest = [nd.get_label() for nd in tmain.extract_subtree(nd_interest).traverse_leaves()]

In [110]:
samples_interest = []
samples_outside = []
for sample in samples:
    if sample.population in population_interest:
        samples_interest.append(sample)
    else:
        samples_outside.append(sample)

In [111]:
time_stop = tmain.extract_subtree(nd_interest.get_parent()).height()
time_interest = tmain.extract_subtree(nd_interest).height()

In [139]:
demography.num_populations

651

In [16]:
demography.sort_events()
for ix, pop in enumerate(demography.populations):
    name_to_pop[ix] = pop.name

In [140]:
tseqAp = msprime.sim_ancestry(
    samples=samples_interest,
    ploidy=2,
    demography=demography,
    recombination_rate=R,
    sequence_length=SL,
    random_seed=(hash(1) % 100000),
    record_migrations=False,
    end_time = time_interest
)

tseqA = msprime.sim_ancestry(
    initial_state=tseqAp,
    ploidy=2,
    demography=demography,
    recombination_rate=R/100000,
    sequence_length=SL,
    random_seed=(hash(1) % 100000),
    record_migrations=False,
    start_time = time_interest,
    end_time = time_stop
)
tseqB = msprime.sim_ancestry(
    samples=samples_outside,
    ploidy=2,
    demography=demography,
    recombination_rate=R,
    sequence_length=SL,
    random_seed=(hash(1) % 100000),
    end_time = time_stop,
    record_migrations=False,
)

The provenance information for the resulting tree sequence is 2.18MB. This is nothing to worry about as provenance is a good thing to have, but if you want to save this memory/storage space you can disable provenance recording by setting record_provenance=False
The provenance information for the resulting tree sequence is 2.18MB. This is nothing to worry about as provenance is a good thing to have, but if you want to save this memory/storage space you can disable provenance recording by setting record_provenance=False
The provenance information for the resulting tree sequence is 2.21MB. This is nothing to worry about as provenance is a good thing to have, but if you want to save this memory/storage space you can disable provenance recording by setting record_provenance=False


In [142]:
tablesB = tseqB.dump_tables()
tablesA = tseqA.dump_tables()

tablesB.edges.child += tseqA.num_nodes
tablesB.edges.parent += tseqA.num_nodes
tablesB.nodes.individual += tseqA.num_individuals

individuals_dict = tablesB.individuals.asdict()
del individuals_dict["metadata_schema"]
tablesA.individuals.append_columns(**individuals_dict)
nodes_dict = tablesB.nodes.asdict()
del nodes_dict["metadata_schema"]
tablesA.nodes.append_columns(**nodes_dict)
edges_dict = tablesB.edges.asdict()
del edges_dict["metadata_schema"]
tablesA.edges.append_columns(**edges_dict)
tablesA.sort()
tablesA.build_index()
ts_joint = tablesA.tree_sequence()

In [143]:
tsC = msprime.sim_ancestry(
    initial_state=ts_joint,
    ploidy=2,
    demography=demography,
    recombination_rate=R,
    sequence_length=SL,
    random_seed=(hash(1) % 100000),
    record_migrations=False,
    start_time = time_stop,
)


The provenance information for the resulting tree sequence is 2.18MB. This is nothing to worry about as provenance is a good thing to have, but if you want to save this memory/storage space you can disable provenance recording by setting record_provenance=False


In [147]:
tsC.simplify()

In [17]:
demography.sort_events()
tseq = msprime.sim_ancestry(
    samples=samples,
    ploidy=2,
    demography=demography,
    recombination_rate=R,
    sequence_length=SL,
    random_seed=(hash(1) % 100000),
    record_migrations=False,
)

for ix, pop in enumerate(demography.populations):
    name_to_pop[ix] = pop.name

The provenance information for the resulting tree sequence is 2.23MB. This is nothing to worry about as provenance is a good thing to have, but if you want to save this memory/storage space you can disable provenance recording by setting record_provenance=False


In [18]:
tseq.simplify()

In [19]:
tables = tseq.dump_tables()

In [20]:
t.encode_bipartitions()
nd_to_pop = {}
gtree_l = []
for name, nd in enumerate(tables.nodes):
    nd_to_pop[name] = tables.populations[nd.population].metadata["name"]
retained_pop_s = set(nd_to_pop.values())
for i in range(0, tseq.get_num_trees()):
    gtree_l.append(tseq.at(i).as_newick(node_labels=nd_to_pop))

In [21]:
with open("example_true.gtrees", "w") as f:
    f.write("\n".join(gtree_l))

In [22]:
pop_to_sunit = {}
with open("../substitution-units.tsv", "r") as f:
    ix = 0
    for l in f:
        if ix > 0:
            name, sunit = l.strip().split("\t")
            pop_to_sunit[name] = float(sunit)
        ix += 1

In [23]:
pop_to_ngen = {}
with open("../num-generations.tsv", "r") as f:
    ix = 0
    for l in f:
        if ix > 0:
            name, ngen = l.strip().split("\t")
            ngen = eval(ngen)
            if ngen is not None:
                pop_to_ngen[name] = float(ngen)
        ix += 1

In [40]:
pop_to_srate = {}
mean_srate = 0
ix = 0
for pop, ngen in pop_to_ngen.items():
    pop_to_srate[pop] = pop_to_sunit[pop] / ngen
pop_to_srate["root"] = mean_srate

In [41]:
tmain = ts.read_tree_newick("../main_neoaves-num_generations.tre")
label_to_node = tmain.label_to_node(selection="all")

In [41]:
derived_to_srates = collections.defaultdict(set)
for ancestral, derived in migrated_pairs:
    subtree_derived = tmain.extract_subtree(label_to_node[derived])
    for node in subtree_derived.traverse_postorder(internal=True, leaves=True):
        derived_to_srates[derived].add(pop_to_srate[node.get_label()])

for derived, srates in derived_to_srates.items():
    pop_to_srate[derived] = np.mean(list(srates))

for ancestral, derived in migrated_pairs:
    subtree_derived = tmain.extract_subtree(label_to_node[derived])
    subtree_ancestral = tmain.extract_subtree(label_to_node[ancestral])
    for node in subtree_ancestral.traverse_postorder(internal=True, leaves=True):
        pop_to_srate[node.get_label()] =  np.sum(list(derived_to_srates[derived])) / subtree_ancestral.num_nodes()
    for node in subtree_derived.traverse_postorder(internal=True, leaves=True):
        pop_to_srate[node.get_label()] = pop_to_srate[derived]

NameError: name 'migrated_pairs' is not defined

In [42]:
for pop, ngen in pop_to_ngen.items():
    if pop in retained_pop_s:
        mean_srate += pop_to_srate[pop]
        ix = ix + 1

mean_srate /= ix
pop_to_rmult = {}
for pop, srate in pop_to_srate.items():
    pop_to_rmult[pop] = srate / mean_srate
pop_to_rmult["root"] = 1

In [43]:
pop_to_stime = {}
pop_to_sdur = {}
pop_to_time = {}
pop_to_dur = {}

for nd in tmain.traverse_levelorder(internal=True, leaves=True):
    l = nd.get_edge_length()
    if l is None:
        l = 0
    pop_to_time[nd.get_label()] = tmain.extract_subtree(nd).height() - l
    pop_to_dur[nd.get_label()] = l

for nd in tmain.traverse_postorder(internal=True, leaves=False):
    for nd_child in nd.child_nodes():
        nd_child.set_edge_length(
            nd_child.get_edge_length() * pop_to_rmult[nd_child.get_label()]
        )

h = tmain.height()
for nd, dist in tmain.distances_from_root(
    leaves=True, internal=True, unlabeled=True, weighted=True
):
    hp = h - dist
    pop_to_stime[nd.get_label()] = hp
    pop_to_sdur[nd.get_label()] = nd.get_edge_length()

pop_to_sdur["root"] = 1
pop_to_dur["root"] = 1

In [44]:
tables = tseq.dump_tables()

In [45]:
pop_to_stime[pop], (time - pop_to_time[pop]) * pop_to_rmult[pop]

KeyError: 'root'

In [46]:
def scale_time_wp(name, time):
    safe = True
    pop = name_to_pop[name]
    # return pop_to_stime[pop] + (time - pop_to_time[pop])/pop_to_dur[pop] * pop_to_sdur[pop]
    return pop_to_stime[pop] + (time - pop_to_time[pop]) * pop_to_rmult[pop]

In [47]:
stime_l = [scale_time_wp(i.population, i.time) for i in tables.nodes]
tables.nodes.time = stime_l

In [35]:
ix = 0
for e in tables.edges:
    if (tables.nodes[e.parent].time - tables.nodes[e.child].time) < 0:
        print(e.parent, e.child, tables.nodes[e.parent].time - tables.nodes[e.child].time)
        print(name_to_pop[tables.nodes[e.child].population])
        print(name_to_pop[tables.nodes[e.parent].population])
        ix += 1
print(ix)

0


In [34]:
ix = 0
for e in tables.edges:
    if (tables.nodes[e.parent].time - tables.nodes[e.child].time) < 0:
        print(e.parent, e.child, tables.nodes[e.parent].time - tables.nodes[e.child].time)
        print(name_to_pop[tables.nodes[e.child].population])
        print(name_to_pop[tables.nodes[e.parent].population])
        ix += 1
print(ix)

0


In [48]:
tables.sort()
tables.simplify()
ntseq = tables.tree_sequence()

In [478]:
ntseq.dump("example.ts")